# AI LÀ AI - TACVU1

Phân biệt ảnh thật / giả (Real vs Fake) sử dụng ResNet-34 tích hợp Supervised Contrastive Learning.

## 1. Cấu hình

In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

from pathlib import Path
import random
import time
import zipfile
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torchvision.models as models
from tqdm.auto import tqdm

# Đường dẫn dữ liệu trên Kaggle
DATA_ROOT = Path('/kaggle/input/datasets/khoileeptit/ca2olp26/TACVU1/data')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('data')

# Giải nén private_test.zip sang thư mục làm việc
if not Path('private_test/images').exists():
    zip_path = DATA_ROOT / 'private_test.zip'
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall('.', pwd=b'629436')

EPOCHS = 5
IMAGE_SIZE = 224
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[cấu hình] DATA_ROOT: {DATA_ROOT}')
print(f'[cấu hình] Thiết bị: {DEVICE} | Epochs: {EPOCHS} | Batch size: {BATCH_SIZE}')


## 2. Mô hình và phép biến đổi ảnh

In [ ]:
class SupConLoss(nn.Module):
    """Hàm mất mát Supervised Contrastive Learning."""
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device
        batch_size = features.shape[0]
        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)

        anchor_dot = torch.div(torch.matmul(features, features.T), self.temperature)
        logits_max, _ = torch.max(anchor_dot, dim=1, keepdim=True)
        logits = anchor_dot - logits_max.detach()

        logits_mask = torch.scatter(torch.ones_like(mask), 1, torch.arange(batch_size).view(-1, 1).to(device), 0)
        mask = mask * logits_mask

        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-7)

        mask_sum = mask.sum(1)
        mask_sum = torch.where(mask_sum == 0, torch.ones_like(mask_sum), mask_sum)
        return - (mask * log_prob).sum(1).div(mask_sum).mean()


class ResNet34(nn.Module):
    """ResNet-34 tích hợp Contrastive Head và Phân loại."""
    def __init__(self, num_classes=2, embedding_dim=128, pretrained=True):
        super().__init__()
        weights = models.ResNet34_Weights.DEFAULT if pretrained else None
        backbone = models.resnet34(weights=weights)
        in_features = backbone.fc.in_features

        self.encoder = nn.Sequential(*list(backbone.children())[:-1], nn.Flatten())
        self.projection_head = nn.Sequential(
            nn.Linear(in_features, in_features),
            nn.ReLU(),
            nn.Linear(in_features, embedding_dim)
        )
        self.classifier = nn.Linear(in_features, num_classes)

    def forward(self, x):
        features = self.encoder(x)
        embeddings = nn.functional.normalize(self.projection_head(features), dim=1)
        logits = self.classifier(features)
        return logits, embeddings


# Transforms
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.2), value='random')
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print(f'[mô hình] ResNet34: {sum(p.numel() for p in ResNet34().parameters()):,} tham số')


## 3. Dataset

In [ ]:
def load_image(path: Path, tf):
    with Image.open(path) as image:
        return tf(ImageOps.exif_transpose(image).convert('RGB'))


class FaceImages(Dataset):
    """Đọc ảnh khuôn mặt cho tập Train và Test."""
    def __init__(self, rows, root, tf, labeled=True):
        self.root = Path(root)
        self.tf = tf
        self.labeled = labeled
        if self.labeled:
            self.image_paths = [self.root / 'images' / Path(p).name for p in rows['path']]
            self.labels = rows['label'].astype(int).tolist()
        else:
            self.image_paths = [self.root / 'images' / fn for fn in rows['file_name']]
            self.file_names = rows['file_name'].tolist()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image = load_image(self.image_paths[index], self.tf)
        if self.labeled:
            return image, self.labels[index]
        return image, self.file_names[index]


## 4. Nạp dữ liệu

In [ ]:
train = pd.read_csv(DATA_ROOT / 'train' / 'manifest.csv')
query = pd.DataFrame({
    'file_name': sorted(p.name for p in Path('private_test/images').iterdir() if p.is_file())
})

train_loader = DataLoader(
    FaceImages(train, DATA_ROOT / 'train', train_transforms), batch_size=BATCH_SIZE, shuffle=True
)

print(f'[dữ liệu] train: {len(train):,} ảnh | phân bố: {train.label.value_counts().to_dict()}')
print(f'[dữ liệu] private_test: {len(query):,} ảnh cần dự đoán')


## 5. Huấn luyện

In [ ]:
model = ResNet34(num_classes=2, embedding_dim=128, pretrained=True).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)
ce_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
con_loss_fn = SupConLoss(temperature=0.07)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, correct, seen = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')

    for images, labels in pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits, embeddings = model(images)
        loss = ce_loss_fn(logits, labels) + 0.1 * con_loss_fn(embeddings, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        seen += labels.size(0)

        pbar.set_postfix({
            'loss': f'{total_loss / seen:.4f}',
            'acc': f'{correct / seen:.4f}'
        })

    scheduler.step()
    lr = scheduler.get_last_lr()[0]
    print(f'[huấn luyện] Epoch {epoch}/{EPOCHS} hoàn tất | Loss: {total_loss / seen:.4f} | Acc: {correct / seen:.4f} | LR: {lr:.6f}')

print('[huấn luyện] hoàn tất')


## 6. Dự đoán và đóng gói bài nộp

In [ ]:
model.eval()
rows = []
started = time.time()

with torch.inference_mode():
    loader = DataLoader(
        FaceImages(query, Path('private_test'), val_transforms, labeled=False), batch_size=BATCH_SIZE
    )
    for images, names in tqdm(loader, desc='Dự đoán private_test'):
        logits, _ = model(images.to(DEVICE))
        labels = logits.argmax(1).cpu().tolist()
        rows.extend(zip(names, labels))

submission = pd.DataFrame(rows, columns=['file_name', 'category_id'])
submission.to_csv('submission.csv', index=False)

with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write('submission.csv')

print(f'[dự đoán] {len(submission):,} ảnh trong {time.time() - started:.1f}s')
print(f'[dự đoán] phân bố nhãn: {submission.category_id.value_counts().to_dict()}')
print('[nộp bài] đã tạo submission.zip (chứa đúng submission.csv ở thư mục gốc)')
